# 사전 실험 A-2: CCTV형 인코딩 + 낮은 비트레이트 재실험
- Input: 데이터셋 구축 노트북 Output (`uvg_cctv`, 무손실 기준 영상 재사용 → 다운로드 없음)
- 인코딩 변경: 2-pass·medium·B프레임 → **실시간 카메라형** 1-pass CBR(VBV = 1초), veryfast, zerolatency(B프레임 없음), GOP 250
- 단계: 4 / 2 / 1 / 0.5 / **0.25 / 0.125** Mbps (+ 비교용 기존 2-pass 0.5 Mbps)
- 설정: GPU T4, Internet On, Save & Run All → Logs + `results2_*.csv` 전달

In [ ]:
!pip install -q ultralytics==8.4.165 pycocotools

In [ ]:
%%writefile eval_lib.py
"""UVG-VCM 비트레이트 단계별 YOLO 평가용 함수"""
import json, os, subprocess, collections
import numpy as np

W, H = 1280, 720
LEVELS = ["ref", "4000k", "2000k", "1000k", "500k"]
CCTV_CLASSES = [1, 2, 3, 4, 6, 8]  # person, bicycle, car, motorcycle, bus, truck (class_id = COCO 순서 + 1)


def build_gt(raw, n_ref_frames, coco_names):
    """UVG-VCM 원본 라벨 -> 720p·30fps COCO GT
    원본 키 k(1부터) = 원본 프레임 k-1, 기준 영상 프레임 i = 원본 프레임 2i -> 키 2i+1"""
    images, anns, aid = [], [], 0
    for i in range(n_ref_frames):
        images.append({"id": i, "file_name": f"{i:06d}.png", "width": W, "height": H})
        for o in raw.get(str(2 * i + 1), []):
            x1, y1 = o["x_min"] * W, o["y_min"] * H
            x2, y2 = o["x_max"] * W, o["y_max"] * H
            w, h = x2 - x1, y2 - y1
            if w <= 0 or h <= 0:
                continue
            aid += 1
            anns.append({"id": aid, "image_id": i, "category_id": int(o["class_id"]),
                         "bbox": [round(x1, 2), round(y1, 2), round(w, 2), round(h, 2)],
                         "area": round(w * h, 2), "iscrowd": int(o.get("iscrowd", 0)),
                         "track_id": o.get("track_id")})
    cats = [{"id": i + 1, "name": n} for i, n in enumerate(coco_names)]
    return {"images": images, "annotations": anns, "categories": cats}


def gt_stats(gt, classes=None):
    a = [x for x in gt["annotations"] if classes is None or x["category_id"] in classes]
    s = sum(x["area"] < 32 ** 2 for x in a); m = sum(32 ** 2 <= x["area"] < 96 ** 2 for x in a)
    names = {c["id"]: c["name"] for c in gt["categories"]}
    cnt = collections.Counter(names[x["category_id"]] for x in a)
    return dict(boxes=len(a), small=s, medium=m, large=len(a) - s - m,
                small_ratio=round(s / max(1, len(a)), 3), top=dict(cnt.most_common(5)))


def extract_frames(video, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    r = subprocess.run(f"ffmpeg -y -loglevel error -i \"{video}\" -start_number 0 \"{out_dir}/%06d.png\"",
                       shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stderr[-2000:])
    return sorted(os.listdir(out_dir))


def filter_preds(preds, max_det=300, conf=0.001):
    by = collections.defaultdict(list)
    for p in preds:
        if p["score"] >= conf:
            by[p["image_id"]].append(p)
    out = []
    for v in by.values():
        v.sort(key=lambda x: -x["score"]); out.extend(v[:max_det])
    return out


def coco_eval(gt, preds, classes=None, img_ids=None):
    """pycocotools bbox 평가. classes: 평가할 category_id 목록 (None이면 GT에 있는 전체)"""
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    import contextlib, io
    g = COCO(); g.dataset = gt
    with contextlib.redirect_stdout(io.StringIO()):
        g.createIndex()
    present = sorted({a["category_id"] for a in gt["annotations"]})
    cat_ids = [c for c in present if classes is None or c in classes]
    ps = [p for p in filter_preds(preds) if p["category_id"] in cat_ids]
    if not ps or not cat_ids:
        return None
    with contextlib.redirect_stdout(io.StringIO()):
        d = g.loadRes(ps)
        E = COCOeval(g, d, "bbox")
        E.params.catIds = cat_ids
        if img_ids is not None:
            E.params.imgIds = img_ids
        E.evaluate(); E.accumulate(); E.summarize()
    names = ["AP", "AP50", "AP75", "APs", "APm", "APl", "AR1", "AR10", "AR100", "ARs", "ARm", "ARl"]
    return {k: round(float(v) * 100, 2) for k, v in zip(names, E.stats)}


In [ ]:
%%writefile build_lib.py
"""UVG-VCM -> CCTV 규격 비트레이트 통제 데이터셋 구축 (Kaggle용)

시퀀스별 출력 ({OUT}/{seq}/):
  ref_720p30.mkv          무손실(FFV1) 기준 영상: 4K60 원본 -> 짝수 프레임만(30fps) -> 1280x720, yuv420p 8bit
  h264_{4000,2000,1000,500}k.mp4   기준 영상에서 각각 한 번씩 압축 (libx264, 2-pass, GOP 250)
  ann_raw.json            UVG-VCM 원본 라벨 그대로
  ann_720p30_coco.json    720p·30fps 기준 COCO 라벨 (변환 가능할 때)
  first.png / last.png    첫·마지막 프레임 미리보기 (고정 시점 확인용)
전체: {OUT}/meta.csv  (시퀀스별 비트 깊이, 단계별 실측 비트레이트·bpp, 라벨 통계)
"""
import csv, json, os, re, subprocess, collections
import numpy as np

W_SRC, H_SRC, FPS_SRC = 3840, 2160, 60
W_OUT, H_OUT, FPS_OUT = 1280, 720, 30
BITRATES_K = [4000, 2000, 1000, 500]
GOP = 250
SCALE = W_SRC / W_OUT  # 3.0


def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"FAILED: {cmd}\n{r.stderr[-3000:]}")
    return r.stdout


def detect_pixfmt(first_frame_u16):
    mx = int(first_frame_u16.max())
    if mx <= 1023:
        return "yuv444p10le", 10, mx
    if mx <= 4095:
        return "yuv444p12le", 12, mx
    return "yuv444p16le", 16, mx


def make_reference(src_cmd, pixfmt, out_mkv, w=W_SRC, h=H_SRC, fps=FPS_SRC):
    """src_cmd: 원본 raw yuv를 stdout으로 내보내는 명령 (curl 등)"""
    vf = (f"select='not(mod(n\\,2))',setpts=N/({FPS_OUT}*TB),"
          f"scale={W_OUT}:{H_OUT}:flags=area,format=yuv420p")
    sh(f"{src_cmd} | ffmpeg -y -loglevel error -f rawvideo -pix_fmt {pixfmt} -s {w}x{h} -framerate {fps} -i pipe:0 "
       f"-vf \"{vf}\" -r {FPS_OUT} -c:v ffv1 -level 3 -pix_fmt yuv420p \"{out_mkv}\"")


def encode_h264(ref_mkv, kbps, out_mp4, tmpdir):
    common = (f"-c:v libx264 -preset medium -profile:v high -pix_fmt yuv420p -b:v {kbps}k "
              f"-g {GOP} -keyint_min {GOP} -sc_threshold 0 -an")
    plog = os.path.join(tmpdir, f"x264_{kbps}")
    sh(f"ffmpeg -y -loglevel error -i \"{ref_mkv}\" {common} -pass 1 -passlogfile \"{plog}\" -f null /dev/null")
    sh(f"ffmpeg -y -loglevel error -i \"{ref_mkv}\" {common} -pass 2 -passlogfile \"{plog}\" \"{out_mp4}\"")


def probe(path):
    js = json.loads(sh(f"ffprobe -v error -select_streams v:0 -count_frames "
                       f"-show_entries stream=codec_name,width,height,nb_read_frames,avg_frame_rate,pix_fmt "
                       f"-show_entries format=duration,size,bit_rate -of json \"{path}\""))
    s, f = js["streams"][0], js["format"]
    n, d = map(float, s["avg_frame_rate"].split("/"))
    fps = n / d
    frames = int(s.get("nb_read_frames", 0))
    dur = frames / fps if frames else float(f["duration"])
    kbps = float(f["size"]) * 8 / dur / 1000  # 파일 크기 기준 실측
    return dict(codec=s["codec_name"], width=s["width"], height=s["height"], fps=round(fps, 3),
                frames=frames, kbps=round(kbps, 1), bpp=round(kbps * 1000 / (s["width"] * s["height"] * fps), 5),
                pix_fmt=s.get("pix_fmt"))


def keyframe_count(path):
    out = sh(f"ffprobe -v error -select_streams v:0 -show_entries frame=pict_type -of csv=p=0 \"{path}\"")
    t = [x.strip().strip(",") for x in out.split() if x.strip().strip(",")]
    return dict(I=t.count("I"), P=t.count("P"), B=t.count("B"))


# ---------------------------------------------------------------- 라벨 변환
_FRAME_KEYS = ("frame_id", "frame_index", "frame_idx", "frame", "frame_number", "frame_no")


def _frame_of(img, order_idx):
    for k in _FRAME_KEYS:
        if k in img and isinstance(img[k], (int, float)):
            return int(img[k]), k
    fn = str(img.get("file_name", ""))
    nums = re.findall(r"\d+", os.path.basename(fn))
    if nums:
        return int(nums[-1]), "file_name"
    return order_idx, "order"


def convert_ann(raw, n_src_frames):
    """COCO형 라벨을 짝수 프레임(30fps)·720p 좌표로 변환. 형식이 다르면 None과 사유 반환"""
    if not (isinstance(raw, dict) and "images" in raw and "annotations" in raw):
        return None, "COCO 형식 아님 (images/annotations 키 없음)"
    imgs = sorted(raw["images"], key=lambda x: x["id"])
    fr, src = {}, collections.Counter()
    for i, im in enumerate(imgs):
        f, how = _frame_of(im, i); fr[im["id"]] = f; src[how] += 1
    frames = sorted(set(fr.values()))
    base = frames[0] if frames else 0  # 0 또는 1부터 시작하는 경우 모두 처리
    if base not in (0, 1):
        return None, f"프레임 번호 시작값 이상: {base}"
    note = f"frame source={dict(src)}, base={base}, frames {len(frames)} (원본 {n_src_frames})"
    new_imgs, idmap = [], {}
    for im in imgs:
        f = fr[im["id"]] - base
        if f % 2:  # 홀수 프레임 버림 (60->30fps)
            continue
        nid = f // 2
        idmap[im["id"]] = nid
        new_imgs.append({"id": nid, "file_name": f"{nid:06d}.png", "width": W_OUT, "height": H_OUT,
                         "src_frame": f})
    anns, k = [], 0
    for a in raw["annotations"]:
        if a.get("image_id") not in idmap or "bbox" not in a:
            continue
        x, y, w, h = [v / SCALE for v in a["bbox"]]
        if w <= 0 or h <= 0:
            continue
        k += 1
        anns.append({"id": k, "image_id": idmap[a["image_id"]], "category_id": a["category_id"],
                     "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
                     "area": round(w * h, 2), "iscrowd": a.get("iscrowd", 0),
                     **({"track_id": a["track_id"]} if "track_id" in a else {})})
    out = {"images": sorted(new_imgs, key=lambda x: x["id"]), "annotations": anns,
           "categories": raw.get("categories", [])}
    return out, note


def ann_stats(coco):
    s = m = l = 0
    for a in coco["annotations"]:
        ar = a["area"]
        s += ar < 32 ** 2; m += 32 ** 2 <= ar < 96 ** 2; l += ar >= 96 ** 2
    cats = {c["id"]: c["name"] for c in coco["categories"]}
    cnt = collections.Counter(cats.get(a["category_id"], a["category_id"]) for a in coco["annotations"])
    n = max(1, len(coco["annotations"]))
    return dict(label_images=len(coco["images"]), label_imgs_with_box=len({a["image_id"] for a in coco["annotations"]}),
                boxes=len(coco["annotations"]), small=s, medium=m, large=l, small_ratio=round(s / n, 3),
                top_classes=json.dumps(dict(cnt.most_common(6)), ensure_ascii=False))


def preview(ref_mkv, out_dir, n_frames):
    sh(f"ffmpeg -y -loglevel error -i \"{ref_mkv}\" -vf \"select='eq(n\\,0)'\" -frames:v 1 \"{out_dir}/first.png\"")
    last = max(0, n_frames - 1)
    sh(f"ffmpeg -y -loglevel error -i \"{ref_mkv}\" -vf \"select='eq(n\\,{last})'\" -frames:v 1 \"{out_dir}/last.png\"")
    from PIL import Image
    a = np.asarray(Image.open(f"{out_dir}/first.png").convert("L").resize((320, 180)), np.float32)
    b = np.asarray(Image.open(f"{out_dir}/last.png").convert("L").resize((320, 180)), np.float32)
    return round(float(np.abs(a - b).mean()), 2)


In [ ]:
import os, sys, json, glob, shutil, time, subprocess
import numpy as np, pandas as pd, torch
sys.path.insert(0, "/kaggle/working"); import eval_lib as E, build_lib as B
from ultralytics import YOLO
assert torch.cuda.is_available(), "GPU 없음"
ROOT = glob.glob("/kaggle/input/**/uvg_cctv", recursive=True)[0]
OUT = "/kaggle/working"; ENC = f"{OUT}/cctv_enc"; TMP = "/tmp/frames"; os.makedirs(ENC, exist_ok=True)
SEQS = ["HighwayView", "CampusView", "JobFair", "VolleyballGame", "FloorballGame", "FloorballTrain"]
KBPS = [4000, 2000, 1000, 500, 250, 125]
LEVELS = ["ref"] + [f"cctv_{k}k" for k in KBPS] + ["2pass_500k"]

def encode_cctv(ref, kbps, out):
    cmd = (f'ffmpeg -y -loglevel error -i "{ref}" -c:v libx264 -preset veryfast -tune zerolatency '
           f'-profile:v high -pix_fmt yuv420p -b:v {kbps}k -maxrate {kbps}k -bufsize {kbps}k '
           f'-g {B.GOP} -keyint_min {B.GOP} -sc_threshold 0 -an "{out}"')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode: raise RuntimeError(r.stderr[-2000:])

rates = []
for s in SEQS:
    ref = f"{ROOT}/{s}/ref_720p30.mkv"; os.makedirs(f"{ENC}/{s}", exist_ok=True)
    for k in KBPS:
        mp4 = f"{ENC}/{s}/cctv_{k}k.mp4"; encode_cctv(ref, k, mp4)
        p = B.probe(mp4); kf = B.keyframe_count(mp4)
        rates.append({"seq": s, "target_kbps": k, "kbps": p["kbps"], "bpp": p["bpp"], "IPB": f"{kf['I']}/{kf['P']}/{kf['B']}"})
rt = pd.DataFrame(rates); rt.to_csv(f"{OUT}/results2_bitrate.csv", index=False)
print(rt.pivot(index="seq", columns="target_kbps", values="kbps").to_string())
print(rt.pivot(index="seq", columns="target_kbps", values="bpp").to_string())

In [ ]:
model = YOLO("yolo26s.pt"); NAMES = [model.names[i] for i in range(80)]
GT = {}
for s in SEQS:
    meta = json.load(open(f"{ROOT}/{s}/done.json"))
    GT[s] = E.build_gt(json.load(open(f"{ROOT}/{s}/ann_raw.json")), meta["ref_frames"], NAMES)

def video_of(s, lv):
    if lv == "ref": return f"{ROOT}/{s}/ref_720p30.mkv"
    if lv == "2pass_500k": return f"{ROOT}/{s}/h264_500k.mp4"
    return f"{ENC}/{s}/{lv}.mp4"

PRED = {}; t0 = time.time()
for s in SEQS:
    for lv in LEVELS:
        fd = f"{TMP}/{s}_{lv}"; files = E.extract_frames(video_of(s, lv), fd)
        assert len(files) == len(GT[s]["images"])
        preds = []
        for i in range(0, len(files), 16):
            chunk = files[i:i + 16]
            for f, r in zip(chunk, model.predict([f"{fd}/{f}" for f in chunk], imgsz=640, conf=0.001,
                                                 max_det=300, device=0, verbose=False)):
                b = r.boxes
                for (x1, y1, x2, y2), c, k in zip(b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy()):
                    preds.append({"image_id": int(f[:6]), "category_id": int(k) + 1,
                                  "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(c)})
        PRED[(s, lv)] = preds; shutil.rmtree(fd)
        print(f"{s:15s} {lv:11s} preds {len(preds):6d} ({time.time()-t0:.0f}s)")

In [ ]:
def merged(seqs, lv):
    g = {"images": [], "annotations": [], "categories": GT[seqs[0]]["categories"]}; p = []; aid = 0
    for j, s in enumerate(seqs):
        off = j * 100000
        g["images"] += [{**im, "id": im["id"] + off} for im in GT[s]["images"]]
        for a in GT[s]["annotations"]:
            aid += 1; g["annotations"].append({**a, "id": aid, "image_id": a["image_id"] + off})
        p += [{**x, "image_id": x["image_id"] + off} for x in PRED[(s, lv)]]
    return g, p

BALL = [33]   # sports ball
GROUPS = [
    ("주평가: 작은객체 CCTV (Highway+Campus)", ["HighwayView", "CampusView"], E.CCTV_CLASSES),
    ("주평가: 공 sports ball (Volley+Floorball)", ["VolleyballGame", "FloorballGame", "FloorballTrain"], BALL),
    ("참고: 고정시점5 CCTV", ["HighwayView", "JobFair", "VolleyballGame", "FloorballGame", "FloorballTrain"], E.CCTV_CLASSES),
    ("참고: 고정시점5 전체", ["HighwayView", "JobFair", "VolleyballGame", "FloorballGame", "FloorballTrain"], None),
] + [(f"{s} CCTV", [s], E.CCTV_CLASSES) for s in ["HighwayView", "CampusView"]]
rows = []
for name, seqs, cls in GROUPS:
    for lv in LEVELS:
        g, p = merged(seqs, lv); r = E.coco_eval(g, p, classes=cls)
        if r: rows.append({"group": name, "level": lv, **r})
df = pd.DataFrame(rows).replace(-100.0, np.nan); df.to_csv(f"{OUT}/results2_all.csv", index=False)
show = ["AP", "AP50", "APs", "APm", "APl", "ARs"]
for grp, sub in df.groupby("group", sort=False):
    t = sub.set_index("level")[show]
    print(f"\n=== {grp} ===\n{t.to_string()}\n원본 대비(%):\n{(t / t.loc['ref'].replace(0, np.nan) * 100).round(1).to_string()}")